# Stage 4, tier 2: BioLangFusion Phase B, one encoder swapped at a time

This notebook asks whether changing one pretrained sequence encoder improves prediction of a measured biological outcome, and whether combining its representation with the other two streams adds value beyond using a single encoder. Those are separate questions: a stronger replacement can improve the whole model even when fusion contributes no further gain.

Stage 1 compared information accessible in frozen encoder representations and similarities between them. [Phase A](tier2_blf_phaseA.ipynb) established the original BioLangFusion comparison. Phase B keeps that local prediction pipeline and evaluation in place while replacing one encoder at a time. It examines H2, the project hypothesis linking representation similarity with useful complementarity; similarity alone does not establish that fusion will help.

The reading route follows that investigation: identify the inputs and replacements, prepare or reuse token embeddings, fit the single-encoder and fusion comparisons, then distinguish score improvements from additional fusion benefit. RNA-FM versus mRNA-FM provides a worked example within the larger sweep.

**Project context:** research plan item 1, Phase B; [Gantt and Details task 24](../../docs/Research_Project_Plan_Gantt.xlsx); Overleaf Section 12.2, item 1. The [Stage 4 overview](README.md#phase-b-changing-one-encoder-at-a-time) connects this comparison to the surrounding experiments.

## Inputs and the biological questions

Each benchmark row pairs a supplied nucleotide sequence with a target and a training, validation, or test assignment. The targets describe different assays; the shared `Value` column does not give them common biological units.

| Benchmark | Target being predicted | Meaning of a better prediction |
| --- | --- | --- |
| [CoV-Vac](../../docs/methods/data-and-targets.md#cov-vac-chemical-degradation-of-designed-rna) | Standardized chemical degradation of designed RNA | Better ordering of the supplied degradation scores; higher labels mean greater degradation |
| [Fungal expression](../../docs/methods/data-and-targets.md#fungal-expression) | Log RNA-seq expression | Better ordering of RNA abundance, not directly measured protein production |
| [E. coli expression](../../docs/methods/data-and-targets.md#protein-expression-in-e-coli) | Low, medium, or high protein-expression class from gel scoring | More correct expression-category predictions in the E. coli host |
| [mRNA stability](../../docs/methods/data-and-targets.md#mrna-stability-standardized-decay-profiles) | Decay slope standardized within its experimental profile and oriented toward stability | Better ordering of relative stability scores, not prediction of lifetime in hours |

The loader retains the supplied split assignments and drops sequences longer than 999 nucleotides; it does not truncate them to 999. This inherited choice reproduces the local BioLangFusion comparison rather than establishing an optimal biological sequence length. See the [Stage 3 dataset audit](../stage3/01_blf_datasets.ipynb) and [preprocessing and evaluation limits](../../docs/methods/data-and-targets.md#splits-preprocessing-and-the-claim-being-tested).

### From one supplied sequence to three encoder streams

DNA and RNA encoders read alphabet-specific versions of the same nucleotide string, including T-to-U conversion for the RNA view. The protein stream is computed by translating complete triplets from the first supplied base, representing stop codons as `X` to retain the positional grid. It is not an independently measured protein sequence. This matters especially for CoV-Vac: its RNA fragments are not verified coding sequences, so the computed protein input should not be described as their demonstrated translation product. [Input construction](../shared_code/fusion/token_embeddings.py)

An encoder writes a numerical feature vector at each token position. Tokens may represent one nucleotide, a three-base codon, a six-base segment, or one amino-acid residue. These token embeddings retain positions, unlike the one-vector-per-sequence pooled embeddings used in the mRFP ridge analysis. A trainable TextCNN prediction head scans neighboring token vectors to predict the target; its kernel size is the number of neighboring positions each filter reads.

The **codon grid** puts the streams on corresponding triplet positions before fusion. Nucleotide-level states are averaged in threes; six-base DNA states use a learned upsampling operation; codon and residue states already have the required spacing. This is computational correspondence under the supplied frame, not proof of biological translation. The replacement table makes each conversion explicit.

### What changes and what stays fixed

Each comparison replaces the encoder in one input slot and retains the other two encoders. It preserves the benchmark rows, split assignments, grid method, prediction-head recipe, and evaluation. Encoder width and token resolution can still change, so this is not an isolation of pretraining data alone.

| Slot | Original encoder | Replacement | Placement on the codon grid |
| --- | --- | --- | --- |
| DNA | Nucleotide Transformer v2 100M | NT 500M human-ref | 6-mers, using the same transposed convolution |
| DNA | Nucleotide Transformer v2 100M | HyenaDNA large | Nucleotide tokens averaged in threes |
| RNA | RNA-FM | RiNALMo | Nucleotide tokens averaged in threes |
| RNA | RNA-FM | mRNA-FM, CaLM | Codon tokens used directly |
| Protein | ESM-2 8M | ESM-2 35M, ESM-2 150M, ProtBERT | Residue tokens used directly |

**Why these comparisons.** The plan names CaLM, mRNA-FM, ProtBERT, ESM-2 150M and alternative DNA encoders. RiNALMo and ESM-2 35M extend the comparison to encoders that Stage 1 found similar to those they replace. Under H2, that similarity motivates testing whether they add less useful information; it does not establish the outcome. DNABERT-2 remains outside this sweep because its variable-length byte-pair tokens need a separately designed, fixed grid adapter.

## RNA-FM and mRNA-FM as a worked encoder comparison

This replacement helps connect the biological inputs, pretrained models and fusion results. Both encoders receive the same supplied nucleotide sequence in the RNA alphabet. The question is whether their different representations change prediction, and whether the DNA and protein streams then add value beyond the replacement alone. The interpretation below was developed from the retained results; it does not reconstruct every historical reason for choosing the models.

### 1. What each encoder learned from

Noncoding RNAs function without being translated into proteins. A coding sequence (CDS) is the part of an mRNA read in three-nucleotide codons during translation; it excludes the untranslated regions. These are different pretraining populations, although both contain nucleotide sequence patterns.

| Model property | RNA-FM | mRNA-FM |
| --- | --- | --- |
| Reported pretraining population | 23.7 million noncoding RNA sequences from RNAcentral, after removing identical sequences | 45 million coding sequences |
| Token unit | One nucleotide | One nonoverlapping three-nucleotide codon |
| Transformer blocks | 12 | 12 |
| Features per content token | 640 | 1,280 |
| Approximate parameter count | 99 million | 239 million |

The [RNA-FM paper's Methods](https://arxiv.org/html/2204.00300v5) describe masked-nucleotide prediction: recover hidden bases from surrounding sequence. The [authors' model documentation](https://github.com/ml4bio/RNA-FM#foundation-models) identifies the mRNA-FM corpus and codon representation. The inspected author materials do not establish mRNA-FM's exact masking recipe, source-database mixture or organism proportions; RNA-FM's detailed training procedure should not simply be assigned to it.

**What changes in this notebook.** RNA-FM represents individual bases, while mRNA-FM represents triplets aligned from the start of the supplied sequence. Both can distinguish synonymous codons; codon tokens impose that grouping rather than introduce sequence information absent from RNA-FM. For fusion, three consecutive RNA-FM states are averaged onto one grid position, while mRNA-FM already supplies one state per codon. With the 999-nucleotide filter and configured 1,024-token limits, both cover the retained sequence; extra sequence coverage is not the explanation for a difference here.

The replacement also changes encoder capacity and predictor inputs. Native TextCNN kernels of three, four and five tokens cover 3–5 nucleotides with RNA-FM but 9–15 with mRNA-FM, and the wider mRNA-FM vectors increase trainable input-layer dimensions. The separate RNA-FM codon-grid control helps examine positional resolution, but does not isolate the pretraining corpus or model capacity. See the [input conversion and grid construction](../shared_code/fusion/token_embeddings.py) and [prediction head](../shared_code/fusion/textcnn.py).

### 2. What capabilities are supported by evidence?

The RNA-FM paper evaluates task-specific predictors using its representations for RNA structure and interactions, and for mean ribosome load from mRNA 5′ untranslated regions. That last result matters here: noncoding-RNA pretraining does not make RNA-FM inherently unsuitable for an mRNA-related task. These evaluations concern the paper's predictors and datasets, not direct measurements of structure in our benchmark rows. [RNA-FM Results, Figure 6 and Table 5](https://arxiv.org/html/2204.00300v5)

Those results belong to RNA-FM. The inspected paper does not supply an equivalent mRNA-FM evaluation, and the author release's general performance description does not establish a quantitative advantage on our targets. Our [mRFP control](tier1_mrfp_control.ipynb) supplies separate local evidence: mRNA-FM's pooled representations improve prediction of synonymous variants' expression beyond the selected linear composition and folding controls. That uses ridge regression and one reporter protein; the token-level comparisons here test transfer to other assays and a different predictor.

The biological expectation is therefore a hypothesis: a representation learned from CDS may make useful coding-sequence patterns easier for the predictor to use. The settings below retain the experiment that tests this expectation. The result discussion returns to where it holds, where it fails, and what remains unresolved.

---

## Setup

The notebook supports NVIDIA CUDA, Apple MPS, and CPU execution. The device cell reports the selected backend before extraction or fitting. No hardware-specific runtime or peak RAM/VRAM requirement is established here; token-array disk estimates below are not memory requirements or duration forecasts.

### Imports

Use the environment described in the [repository setup instructions](../../README.md#setup). Dependency roles are noted beside their imports; the following cells locate shared project code and report the compute device.

In [24]:
# Files, environment, timestamps, and timing
import datetime
import json
import os
import sys
import time
from pathlib import Path

# Set Apple GPU fallback before importing PyTorch; preserve an explicit user setting.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Numerical arrays and result tables
import numpy as np
import pandas as pd

# Tensor operations, model training, and available compute backends
import torch

# Render notebook tables explicitly rather than relying on an implicit kernel name.
from IPython.display import display

### Project paths and shared code

Locate the repository's `Code` directory so the same shared functions can be imported from the repository root or a notebook folder.

In [25]:
# Search the working directory and its parents for Code/shared_code.
CODE_DIR = next(
    (
        candidate.resolve()
        for parent in (Path.cwd(), *Path.cwd().parents)
        for candidate in (parent, parent / "Code")
        if (candidate / "shared_code").is_dir()
    ),
    None,
)
if CODE_DIR is None:
    raise RuntimeError("Open this repository in VS Code before running this notebook.")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Benchmark rows and encoder definitions
from shared_code import datasets, encoders
from shared_code.encoders import ENCODERS

# Fusion input streams, prediction-head training, and token-cache verification
from shared_code.fusion import blf, train, swap_results, swap_plan
from shared_code.fusion import token_embeddings as tokens

### Compute device

Inspect the device report before extraction or training. Availability depends on the installed PyTorch build and local hardware; this notebook supports Apple MPS, NVIDIA CUDA, and CPU. The report identifies the selected backend, not a runtime or memory estimate.

In [26]:
# Table display width (characters) and visible columns, not sequence or token limits.
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Preserve the existing preference: Apple MPS, then NVIDIA CUDA, then CPU.
DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(
    "torch", torch.__version__,
    "| CUDA available:", torch.cuda.is_available(),
    "| Apple MPS available:", torch.backends.mps.is_available(),
)
if DEVICE == "cuda":
    print(f"Selected device: {DEVICE} | GPU: {torch.cuda.get_device_name()}")
elif DEVICE == "mps":
    print(f"Selected device: {DEVICE} | GPU: Apple GPU (MPS)")
else:
    print(f"Selected device: {DEVICE}")

torch 2.10.0+cu126 | CUDA available: True | Apple MPS available: False
Selected device: cuda | GPU: NVIDIA GeForce RTX 2060 SUPER


### Settings and comparison scope

**Benchmarks.** The configured sweep covers CoV-Vac, fungal expression, E. coli and mRNA stability. Each benchmark now includes all eight replacements. The earlier stability cohort included mRNA-FM, CaLM and ESM-2 150M; its completed results remain in place while the five other replacements are added. This expands the experiment described by task 24 in the [research plan](../../docs/Research_Project_Plan_Gantt.xlsx). Ab1 remains excluded because its data are not public.

**Manual seeds.** Leave `READ_SAVED_RESULTS = True` for inspection. To complete the configured fits, set it to `False` and begin with `SEED = 0`. After that run finishes, manually change `SEED` to `1`, run from the top, then do the same for `2`. The notebook never advances the seed automatically. It reuses compatible completed records for the requested seed and trains only missing entries. Frozen token embeddings are shared across seeds; changing the training seed does not require extracting them again.

**Fixed selection.** Keep `SELECTION_SEED = 0`. The original encoders' complete positive-penalty validation grid selects the gated-head penalty at that seed. Each triple's best-single baseline is also chosen using its seed-0 validation scores, then that same model variant is compared at the requested training seed. Neither choice is retuned for seeds 1 or 2. Repeat preflight therefore requires the seed-0 single-encoder choices to be available before new work starts.

**Where work is saved.** Token caches remain under `Code/stage3/embeddings/<benchmark>/<encoder>/`. Replacement fits use `Code/stage4/results/phaseB/<benchmark>/<encoder>/`. Missing original-encoder references use `Code/stage4/results/phaseA/<benchmark>/`; compatible earlier originals may also be read from Stage 3. Filenames retain the actual training seed. No earlier result is renamed or replaced to fill a missing prediction or metadata field.

**Reading and analysis.** Default Run All reads result evidence and reports missing work without loading datasets or token arrays, fitting, or writing results. Historical identity limits remain explicit. In computation mode, `RUN_BOOTSTRAP = False` keeps interval resampling separate from completing fits, so a repeated pass does not recalculate that analysis. Changing it to `True` explicitly requests the paired-interval calculation; it does not fit models or write over saved fit records.


In [27]:
# Default Run All reads saved evidence; changing this enables research computation.
READ_SAVED_RESULTS = False

BENCHMARKS = (
    "cov_vac_degradation",
    "fungal_expression",
    "ecoli_proteins",
    "mrna_stability",
)
CACHE_ROOT = CODE_DIR / "stage3" / "embeddings"
RESULTS_ROOT = CODE_DIR / "stage4" / "results" / "phaseB"

# Original-encoder results are read for comparison and validation-selected MIL weights.
ORIGINAL_RESULTS = {
    "cov_vac_degradation": [Path("stage4/results/phaseA/cov_vac_degradation")],
    "ecoli_proteins": [Path("stage4/results/phaseA/ecoli_proteins")],
    "fungal_expression": [
        Path("stage3/results/fungal_expression"),
        Path("stage4/results/phaseA/fungal_expression"),
    ],
    "mrna_stability": [Path("stage4/results/phaseA/mrna_stability")],
}
ORIGINAL_RESULTS = {
    key: [CODE_DIR / folder for folder in folders]
    for key, folders in ORIGINAL_RESULTS.items()
}
STAGE1_ALIGNMENT = CODE_DIR.parent / "docs" / "figures" / "stability" / "results.json"

#### Training settings and replacement choices

The pretrained encoders stay frozen. A trainable **TextCNN** scans local windows of token features and summarizes them to predict the target. Here it retains 100 learned filters for each kernel size of three, four, and five positions; the biological span of a position depends on the encoder or grid. The [prediction-head implementation](../shared_code/fusion/textcnn.py) records the shared architecture and the paper's width ambiguity.

`MAX_EPOCHS` is an upper limit on passes through training data, not a promised duration. The shared [training configuration](../shared_code/fusion/train.py) retains validation-based early stopping. `DTYPE` controls stored token arrays; it does not select half-precision encoder loading or training.

In [28]:
# Prediction-head training settings
TCONV = "equation"  # Retain the Equation 1 six-mer-to-codon upsampling.
CHANNELS = 100      # Learned filters per TextCNN kernel size, not encoder width.
BATCH_SIZE = 32     # Benchmark rows per training minibatch.
MAX_EPOCHS = 500    # Ceiling; validation-based early stopping may end a fit sooner.
SEED = 0           # Change manually to 1, then 2; one training seed per execution.
SELECTION_SEED = 0 # Keep validation-selected penalties and baseline choices fixed.
RUN_BOOTSTRAP = False  # Separate analysis; leave off when completing fits or repeats.

# Cache chunking and stored-array precision, not encoder or training precision
CHUNK_SIZE = tokens.DEFAULT_CHUNK
DTYPE = np.float16

# Each pair replaces one slot while retaining the other two original encoders.
SWAPS = [
    ("dna", "nt500m"),
    ("dna", "hyenadna"),
    ("rna", "rinalmo"),
    ("rna", "mrnafm"),
    ("rna", "calm"),
    ("prot", "esm2_35m"),
    ("prot", "esm2_150m"),
    ("prot", "protbert"),
]
# Include all eight replacements; completed fits in the former subset are reused.
STABILITY_SWAPS = list(SWAPS)
ORIGINAL = dict(blf.ENCODER_OF_MODALITY)

#### Select the comparisons and shared caches

`swaps_for` returns the configured replacement list for each benchmark. Stability now uses the same eight replacements as the other benchmarks. `required_encoders` lists the original streams and replacements without duplicates. The execution plan below narrows cache work to encoders needed by genuinely pending fits; completed comparisons do not require their large arrays to be opened.


In [29]:
def swaps_for(key):
    """Use the selected stability subset, or the full swap list for other benchmarks."""
    return STABILITY_SWAPS if key == "mrna_stability" else SWAPS


def required_encoders(key):
    """List the original and replacement encoders needed for one benchmark."""
    # Remove duplicates without changing order: original streams are checked first.
    return list(dict.fromkeys([
        *ORIGINAL.values(),
        *(e for _, e in swaps_for(key)),
    ]))

#### Plan the selected seed

The plan distinguishes reusable completed records, new fits, resumable checkpoints and blocked evidence. It includes original references and replacements separately, then lists entries needing attention. In computation mode every benchmark must pass before any extraction begins. When a checkpoint needs inspection, its recorded recipe and recovery state are checked on CPU; read-only mode does not deserialize it. A plan is a snapshot, not proof of unrecorded historical provenance.


In [30]:
# Plan every requested fit before extraction. Reuse actual saved paths, including
# compatible historical names; a missing result is different from a conflicting file.
def execution_plan(key):
    """Use the same preflight before extraction and immediately before fitting."""
    return swap_plan.plan_benchmark(
        benchmark=key,
        config=vars(train.TrainConfig(
            batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS, channels=CHANNELS,
            tconv=TCONV, seed=SEED,
        )),
        selection_seed=SELECTION_SEED,
        original_encoders=ORIGINAL,
        original_folders=ORIGINAL_RESULTS[key],
        original_destination=CODE_DIR / "stage4" / "results" / "phaseA" / key,
        swaps=[(m, e, tokens.grid_resolution(e) != "codon") for m, e in swaps_for(key)],
        swap_root=RESULTS_ROOT / key,
        checkpoint_validator=None if READ_SAVED_RESULTS else train.checkpoint_preflight,
    )


FIT_PLANS = {key: execution_plan(key) for key in BENCHMARKS}
fit_inventory = pd.DataFrame([row for plan in FIT_PLANS.values() for row in plan.rows()])
if not fit_inventory.empty:
    display(fit_inventory.groupby(["benchmark", "role", "status"]).size().rename("fits"))
    display(fit_inventory.loc[fit_inventory["status"] != "reuse"])
for key, plan in FIT_PLANS.items():
    for issue in plan.issues:
        print(f"{key}: {issue}")

if not READ_SAVED_RESULTS:
    # Stop the entire request before expensive work if any comparison is blocked.
    for plan in FIT_PLANS.values():
        plan.require_ready()

PENDING_RUNS = [
    run for plan in FIT_PLANS.values() for run in plan.runs
    if run.status in ("new", "resume")
]
PENDING_CACHE_KEYS = {
    (run.benchmark, encoder_key)
    for run in PENDING_RUNS for encoder_key in run.encoders.values()
}
print(f"Training seed {SEED}; selection seed {SELECTION_SEED}; "
      f"{len(PENDING_RUNS)} new/resumable fits in the displayed plan.")


benchmark            role         status
cov_vac_degradation  original     reuse      7
                     replacement  reuse     35
ecoli_proteins       original     reuse      7
                     replacement  reuse     35
fungal_expression    original     reuse      7
                     replacement  reuse     35
mrna_stability       original     reuse      7
                     replacement  new       23
                                  reuse     12
Name: fits, dtype: int64

,benchmark,role,encoder,variant,seed,lambda,status,reason,source,destination,legacy identity
133,mrna_stability,replacement,nt500m,unimodal:dna,0,0.00,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False
134,mrna_stability,replacement,nt500m,unimodal-grid:dna,0,0.00,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False
135,mrna_stability,replacement,nt500m,concat,0,0.00,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False
136,mrna_stability,replacement,nt500m,mil,0,0.01,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False
137,mrna_stability,replacement,nt500m,cross,0,0.00,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False
138,mrna_stability,replacement,hyenadna,unimodal:dna,0,0.00,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False
139,mrna_stability,replacement,hyenadna,unimodal-grid:dna,0,0.00,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False
140,mrna_stability,replacement,hyenadna,concat,0,0.00,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False
141,mrna_stability,replacement,hyenadna,mil,0,0.01,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False
142,mrna_stability,replacement,hyenadna,cross,0,0.00,new,No compatible completed result or recovery art...,,D:\academic\UCF\2026_fall\Capstone Project CAP...,False


Training seed 0; selection seed 0; 23 new/resumable fits in the displayed plan.


In [31]:
planned = sum(len(swaps_for(k)) for k in BENCHMARKS)
print(f"{planned} swap-benchmark pairs over {len(BENCHMARKS)} benchmarks, device {DEVICE}")
pd.DataFrame([
    {
        "slot": m,
        "replaces": ENCODERS[ORIGINAL[m]].label,
        "swapped in": ENCODERS[k].label,
        "grid": tokens.grid_resolution(k),
    }
    for m, k in SWAPS
])

32 swap-benchmark pairs over 4 benchmarks, device cuda


,slot,replaces,swapped in,grid
0,dna,NT v2 100M multi-species,NT 500M human-ref,kmer
1,dna,NT v2 100M multi-species,HyenaDNA large,nucleotide
2,rna,RNA-FM,RiNALMo 150M,nucleotide
3,rna,RNA-FM,mRNA-FM,codon
4,rna,RNA-FM,CaLM,codon
5,prot,ESM-2 8M,ESM-2 35M,codon
6,prot,ESM-2 8M,ESM-2 150M,codon
7,prot,ESM-2 8M,ProtBERT,codon


### Estimate token-array storage

This computation-only projection reads rows for benchmarks with pending fits and estimates the selected replacement arrays' payload from token counts, stored dtype and encoder width. It does not extract embeddings. It can include already cached arrays needed by a missing fit, so the total is not additional free space required. Full cache checks below establish which shards actually need extraction. Model downloads, temporary publication space and peak RAM/GPU allocations are not measured by this projection.


In [32]:
if READ_SAVED_RESULTS:
    projection = pd.DataFrame()
    print("Storage projection skipped: reading saved results does not load benchmark inputs.")
elif not PENDING_RUNS:
    projection = pd.DataFrame()
    print("All requested fits are complete; no extraction storage is needed.")
else:
    # Token-array payload only; widths describe the selected checkpoints.
    WIDTHS = {
        "nt500m": 1280,
        "hyenadna": 256,
        "rinalmo": 640,
        "mrnafm": 1280,
        "calm": 768,
        "esm2_35m": 480,
        "esm2_150m": 640,
        "protbert": 1024,
    }

    projection = []
    for key in BENCHMARKS:
        if not any(b == key for b, _ in PENDING_CACHE_KEYS):
            continue
        lengths = datasets.blf_benchmark_rows(key)["length"].to_numpy()
        for modality, encoder_key in swaps_for(key):
            if (key, encoder_key) not in PENDING_CACHE_KEYS:
                continue
            per_row = np.array([
                tokens.tokens_for_length(ENCODERS[encoder_key], int(n))
                for n in lengths
            ])
            # Sum token counts across rows, then multiply by features/token and bytes/feature.
            # Divide by 1e9 for decimal GB; this excludes model and training memory.
            gigabytes = (
                per_row.sum() * WIDTHS[encoder_key] * np.dtype(DTYPE).itemsize / 1e9
            )
            projection.append({
                "benchmark": key,
                "encoder": ENCODERS[encoder_key].label,
                "rows": len(lengths),
                "GB": round(gigabytes, 1),
            })
    projection = pd.DataFrame(projection, columns=["benchmark", "encoder", "rows", "GB"])
    display(
        projection.pivot_table(
            index="benchmark", columns="encoder", values="GB", aggfunc="sum",
        ).fillna("")
    )
    print(f"total token cache for the planned swaps: {projection['GB'].sum():.0f} GB")
    print("All of it stays out of Git, under Code/stage3/embeddings/<benchmark>/<encoder>/.")

encoder,ESM-2 35M,HyenaDNA large,NT 500M human-ref,ProtBERT,RiNALMo 150M
benchmark,,,,,
mrna_stability,5.0,8.0,6.8,10.7,20.1


total token cache for the planned swaps: 51 GB
All of it stays out of Git, under Code/stage3/embeddings/<benchmark>/<encoder>/.


**Reading the storage projection.** The table above multiplies expected content-token counts by a hard-coded hidden width and the cache dtype size. Its decimal GB values cover token-array payload only, excluding model downloads, archive and metadata overhead, training checkpoints and other saved weights. They do not establish which resource will limit a run.

**Width and saved-output boundary.** The projection now uses 640 features per RiNALMo token, matching the inspected token-array headers in the current CoV-Vac, fungal and E. coli caches. The preserved output was produced with the earlier 1,280-width lookup and can therefore show twice the RiNALMo payload. This correction affects a storage estimate, not embeddings or fitted results. Token payload alone remains insufficient to estimate training RAM, VRAM or runtime.

### Check the original encoders' results

Phase B needs these runs both as score references and to select the gated head's entropy weight: Phase A for CoV-Vac, E. coli and mRNA stability; Stage 3 step 3 or the configured Phase A folder for fungal. Check the table below for available variants and a gated lambda before starting training. The fit plan above uses those configured paths to find compatible records. It requires the completed seed-0 tuning grid before work; when repeating a training seed, it schedules that seed's missing original references before replacements.

In [33]:
def label(r):
    """Keep unregularized MIL separate from the selected positive-penalty variant."""
    return (
        "mil (lam 0)"
        if r.variant == "mil" and r.config["entropy_lambda"] == 0
        else r.variant
    )


def result_recipe(seed):
    """Match the full retained TrainConfig, including defaults not exposed above."""
    return vars(train.TrainConfig(
        batch_size=BATCH_SIZE,
        max_epochs=MAX_EPOCHS,
        channels=CHANNELS,
        tconv=TCONV,
        seed=seed,
    ))


def original_report(key, seed):
    return swap_results.read_results(
        ORIGINAL_RESULTS[key],
        benchmark=key,
        config=result_recipe(seed),
        encoders=ORIGINAL,
        native_fallback=True,
    )


def original_runs(key, seed=None):
    """Read the selected seed; ambiguous identities cannot silently replace one another."""
    report = original_report(key, SEED if seed is None else seed)
    # Duplicated identities are withheld by the reader; other saved scores still stand.
    records = report.records if READ_SAVED_RESULTS else report.require_unique()
    return [train.Result.load(record["path"]) for record in records]


def original_lambda(key):
    """Reuse the complete original positive-penalty grid's seed-0 selection."""
    lam = FIT_PLANS[key].lambda_
    if lam is None:
        raise RuntimeError(f"{key}: seed-0 MIL selection unavailable; inspect the execution plan")
    return lam


def swap_report(key, modality, encoder_key, seed, lam):
    """Check the replacement recipe, encoder mapping when recorded, and fit names."""
    expected = swap_results.swap_variants(
        modality, grid=tokens.grid_resolution(encoder_key) != "codon",
    )
    return swap_results.read_results(
        [RESULTS_ROOT / key / encoder_key],
        benchmark=key,
        config=result_recipe(seed),
        encoders={**ORIGINAL, modality: encoder_key},
        mil_lambda=lam,
        variants=expected if lam is not None else tuple(v for v in expected if v != "mil"),
    )

The next cell reads the configured result folders and lists the available variants. A missing gated lambda indicates an unmet prerequisite; this check does not fit a replacement model.

In [34]:
status = []
for key in BENCHMARKS:
    report = original_report(key, SEED)
    have = sorted({label(train.Result.load(r["path"])) for r in report.records})
    try:
        lam = original_lambda(key)
    except RuntimeError as exc:
        # Display the prerequisite failure before any extraction or fitting.
        lam = None
        prerequisite = str(exc)
    else:
        prerequisite = ""
    status.append({
        "benchmark": key,
        "original runs": len(report.records),
        "variants": ", ".join(have),
        "gated lambda": lam,
        "prerequisite issue": prerequisite,
        "excluded files": len(report.issues),
    })
pd.DataFrame(status).set_index("benchmark")

,original runs,variants,gated lambda,prerequisite issue,excluded files
benchmark,,,,,
cov_vac_degradation,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01,,8
fungal_expression,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01,,16
ecoli_proteins,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01,,8
mrna_stability,10,"concat, cross, mil, mil (lam 0), unimodal-grid...",0.01,,0


### Check saved-result coverage before computation

This read-only check separates saved fits from token-cache readiness. It checks the direct result folders for the declared training recipe, identifies excluded or ambiguous records, and shows which comparisons have saved scores and paired prediction arrays. It does not open token caches, load encoders, fit a model, or calculate new scores.

Seeds 0, 1 and 2 are inspected to make missing repeats visible. This inspection does not enable those repeats: fitting and the comparison tables still use `SEED`. Coverage for every inspected seed uses the positive MIL penalty selected by validation at `SELECTION_SEED = 0`; it does not retune the penalty for a future repeat. A replacement also needs the unchanged single-encoder references and original fusion heads from the same inspected seed. This coverage cell only reports missing references; the separate execution plan schedules them when computation is enabled.

“Legacy identity” counts eligible records whose encoder mapping was not recorded. Folder placement is the historical attribution for these files, not a new verification of their inputs or producer. “Without paired arrays” distinguishes an available score from evidence usable for the later bootstrap. The detailed exclusions remain in `result_issues`; the displayed reason counts explain why files were not selected. The check is a snapshot when this cell is run, not a monitor of another process.

Coverage also lists the original RNA-FM grid control when RNA is unchanged, and flags score-only reference records separately. The score reader retains the historical rule that original grid controls are eligible where available; the execution plan includes a missing same-seed control needed to retain that candidate set.

In [35]:
# Inspection only: these seeds do not alter SEED or launch additional fits.
COVERAGE_SEEDS = (0, 1, 2)
coverage_rows, issue_rows = [], []
original_variants = (
    "unimodal:dna", "unimodal:rna", "unimodal:prot", "unimodal-grid:rna",
    "concat", "mil", "cross",
)

for key in BENCHMARKS:
    try:
        inherited_lambda = original_lambda(key)
        prerequisite = ""
    except RuntimeError as exc:
        inherited_lambda = None
        prerequisite = str(exc)

    # Zero here marks unavailable positive-penalty selection; it does not select MIL(0).
    coverage_lambda = inherited_lambda if inherited_lambda is not None else 0.0
    for inspected_seed in COVERAGE_SEEDS:
        originals = original_report(key, inspected_seed)
        # The coverage row concerns the inherited penalty, not another seed's tuning winner.
        original_coverage = originals.coverage(original_variants, mil_lambda=coverage_lambda)
        coverage_rows.append({
            "benchmark": key, "encoder": "original", "seed": inspected_seed,
            **original_coverage, "reference gaps": "", "reference arrays absent": "",
            "prerequisite": prerequisite,
        })
        issue_rows.extend({
            "benchmark": key, "encoder": "original", "seed": inspected_seed, **issue,
        } for issue in originals.issues)

        for modality, encoder_key in swaps_for(key):
            expected = swap_results.swap_variants(
                modality, grid=tokens.grid_resolution(encoder_key) != "codon",
            )
            report = swap_report(
                key, modality, encoder_key, inspected_seed, inherited_lambda,
            )
            references = [
                *(f"unimodal:{m}" for m in ORIGINAL if m != modality),
                *(("unimodal-grid:rna",) if modality != "rna" else ()),
                "concat", "mil", "cross",
            ]
            reference_coverage = originals.coverage(references, mil_lambda=coverage_lambda)
            coverage_rows.append({
                "benchmark": key, "encoder": ENCODERS[encoder_key].label,
                "seed": inspected_seed,
                **report.coverage(expected, mil_lambda=coverage_lambda),
                "reference gaps": reference_coverage["missing"],
                "reference arrays absent": reference_coverage["without paired arrays"],
                "prerequisite": prerequisite,
            })
            issue_rows.extend({
                "benchmark": key, "encoder": encoder_key, "seed": inspected_seed, **issue,
            } for issue in report.issues)

result_coverage = pd.DataFrame(coverage_rows).set_index(["benchmark", "encoder", "seed"])
result_issues = pd.DataFrame(issue_rows)
display(result_coverage)
if not result_issues.empty:
    display(result_issues.groupby(["seed", "reason"]).size().rename("excluded records"))

available                            missing without paired arrays ambiguous  excluded files  legacy identity  \
benchmark           encoder           seed                                                                                                                                                          
cov_vac_degradation original          0     unimodal:dna, unimodal:rna, unimodal:prot, uni...                                                                                  8               10   
                    NT 500M human-ref 0     unimodal:dna, unimodal-grid:dna, concat, mil, ...                                                                                  0                5   
                    HyenaDNA large    0     unimodal:dna, unimodal-grid:dna, concat, mil, ...                                                                                  0                5   
                    RiNALMo 150M      0     unimodal:rna, unimodal-grid:rna, concat, mil, ...                                                                                  0                5   
                    mRNA-FM           0                      unimodal:rna, concat, mil, cross                                                                                  0                0   
...                                                                                       ...                                ...                   ...       ...             ...              ...   
mrna_stability      mRNA-FM           2                                                         unimodal:rna, concat, mil, cross                                               4                0   
                    CaLM              2                                                         unimodal:rna, concat, mil, cross                                               4                0   
                    ESM-2 35M         2                                                        unimodal:prot, concat, mil, cross                                               0                0   
                    ESM-2 150M        2                                                        unimodal:prot, concat, mil, cross                                               4                0   
                    ProtBERT          2                                                        unimodal:prot, concat, mil, cross                                               0                0   

                                                                               reference gaps reference arrays absent prerequisite  
benchmark           encoder           seed                                                                                          
cov_vac_degradation original          0                                                                                             
                    NT 500M human-ref 0                                                                                             
                    HyenaDNA large    0                                                                                             
                    RiNALMo 150M      0                                                                                             
                    mRNA-FM           0                                                                                             
...                                                                                       ...                     ...          ...  
mrna_stability      mRNA-FM           2       unimodal:dna, unimodal:prot, concat, mil, cross                                       
                    CaLM              2       unimodal:dna, unimodal:prot, concat, mil, cross                                       
                    ESM-2 35M         2     unimodal:dna, unimodal:rna, unimodal-grid:rna,...                                       
                    ESM-2 150M        2     unimodal:dna, u

seed  reason                                        
0     configuration differs: channels                     1
      configuration differs: seed                        24
      configuration differs: tconv                        6
      native baseline with requested tconv preferred      1
1     configuration differs: channels, seed               1
      configuration differs: seed                       170
      configuration differs: seed, tconv                  6
2     configuration differs: channels, seed               1
      configuration differs: seed                       170
      configuration differs: seed, tconv                  6
Name: excluded records, dtype: int64

---

## Part 1: Prepare the token embeddings

### Check caches before loading models

Inspect the cache pairs required by pending fits, including their original streams. A fully reused fit plan opens no token arrays. The preflight checks extraction settings, recorded ordered-input hashes, shard structure, consistent widths and each row's token count. Available historical shard hashes are also checked. Verification can be reused between preflight and extraction while file statistics remain unchanged; rerunning this preflight creates a fresh verification object.

**Input identity is required.** Reusing a cache requires recorded ordered-input identity; this notebook has no legacy-cache reuse opt-in. New extraction records that identity before the first shard, and a mismatch stops reuse. The [reset record](../../docs/run-records.md#2026-10-25-encoder-swap-unverified-cache-reset) preserves the history of the legacy-cache cleanup.


In read-saved mode this cell lists only saved cache metadata. It neither reads shards nor establishes cache compatibility; full preflight belongs to the computation branch. Missing metadata is reported without blocking the result tables.


In [36]:
if READ_SAVED_RESULTS:
    # Inspect small metadata files only. Their presence does not verify the shards.
    CACHE_PLANS = {}
    BENCHMARK_ROWS = {}
    checks = []
    for benchmark in BENCHMARKS:
        for encoder_key in required_encoders(benchmark):
            path = CACHE_ROOT / benchmark / encoder_key / "meta.json"
            try:
                metadata = json.loads(path.read_text(encoding="utf-8"))
                if not isinstance(metadata, dict):
                    raise ValueError("metadata is not an object")
                cache_status = "saved metadata; shards not checked"
                issue = ""
            except (OSError, ValueError) as exc:
                metadata = {}
                cache_status = "metadata unavailable"
                issue = str(exc)
            checks.append({
                "benchmark": benchmark,
                "encoder": ENCODERS[encoder_key].label,
                "status": cache_status,
                "recorded shards": metadata.get("n_shards"),
                "issues": issue,
            })
    display(pd.DataFrame(checks).set_index(["benchmark", "encoder"]))
    print("Saved scores remain readable without token caches. No recovery is attempted.")
else:
    # Historical shard evidence is optional; ordered-input identity is still required.
    recovery_path = CACHE_ROOT / "recovery-records.json"
    RECOVERY_RECORDS = (
        {r["key"]: r for r in json.loads(recovery_path.read_text())["caches"]}
        if recovery_path.exists() else {}
    )

    # Share successful file checks with extraction until the file statistics change.
    CACHE_VERIFICATION = tokens.CacheVerification()
    CACHE_PLANS = {}
    BENCHMARK_ROWS = {}
    checks = []
    for benchmark in BENCHMARKS:
        needed = [e for e in required_encoders(benchmark)
                  if (benchmark, e) in PENDING_CACHE_KEYS]
        if not needed:
            continue
        rows = datasets.blf_benchmark_rows(benchmark)
        BENCHMARK_ROWS[benchmark] = rows
        for encoder_key in needed:
            encoder = ENCODERS[encoder_key]
            texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
            plan = tokens.inspect_cache(
                encoder,
                texts,
                CACHE_ROOT / benchmark / encoder_key,
                chunk_size=CHUNK_SIZE,
                dtype=DTYPE,
                recovery_evidence=RECOVERY_RECORDS.get(f"{benchmark}/{encoder_key}"),
                verification=CACHE_VERIFICATION,
            )
            CACHE_PLANS[benchmark, encoder_key] = plan
            checks.append({
                "benchmark": benchmark,
                "encoder": encoder.label,
                "status": plan.status,
                "reusable shards": len(plan.reusable_shards),
                "missing shards": len(plan.missing_shards),
                "input provenance": plan.provenance,
                "issues": "; ".join(plan.issues),
            })
            print(f"  {benchmark}/{encoder_key}: {plan.status}, {plan.provenance}", flush=True)
    if checks:
        display(pd.DataFrame(checks).set_index(["benchmark", "encoder"]))
    else:
        print("All requested fits are complete; cache verification is unnecessary.")

    # Stop before loading models if any cache cannot safely be reused or extended.
    blocked = [plan.summary for plan in CACHE_PLANS.values() if not plan.ready]
    if blocked:
        raise tokens.CacheRecoveryRequired(
            "Cache preflight stopped before model loading:\n" + "\n".join(blocked)
        )
    missing = sum(len(plan.missing_shards) for plan in CACHE_PLANS.values())
    print(f"Preflight passed: {len(CACHE_PLANS)} caches; {missing} shards need extraction.")


  mrna_stability/ntv2_100m: complete, recorded_input
  mrna_stability/rnafm: complete, recorded_input
  mrna_stability/esm2_8m: complete, recorded_input
  mrna_stability/nt500m: new, new
  mrna_stability/hyenadna: new, new
  mrna_stability/rinalmo: new, new
  mrna_stability/esm2_35m: new, new
  mrna_stability/protbert: new, new


status  reusable shards  missing shards input provenance issues
benchmark      encoder                                                                                    
mrna_stability NT v2 100M multi-species  complete               94               0   recorded_input       
               RNA-FM                    complete               94               0   recorded_input       
               ESM-2 8M                  complete               94               0   recorded_input       
               NT 500M human-ref              new                0              94              new       
               HyenaDNA large                 new                0              94              new       
               RiNALMo 150M                   new                0              94              new       
               ESM-2 35M                      new                0              94              new       
               ProtBERT                       new                0              94              new

Preflight passed: 8 caches; 470 shards need extraction.


**Reading the cache-status table.** Each row describes one benchmark and encoder cache. A shard is a saved chunk of token embeddings; shard counts describe stored inputs rather than trained models.

| Field | Meaning |
| --- | --- |
| `status` | `complete` means all expected shards are present and the preflight reports no issues. It describes embedding readiness, not completion of downstream training. |
| `reusable shards` | Existing chunks that passed the applicable reuse checks. |
| `missing shards` | Expected chunks that are absent and would need extraction. |
| `input provenance` | `recorded_input` means the cache stores an identity for its ordered input sequences. Successful preflight with no issues establishes that this identity matches the requested inputs. |
| `issues` | Problems that block reuse and need inspection before extraction or training. |

When every required cache is complete, has no missing shards and reports no issues, no new embedding computation is needed. The reuse cell below opens only the caches needed by pending fits and reports what it found.

### Extract missing shards and reuse complete caches

For pending fits, open complete caches and compute only missing chunks permitted by the preflight. The original streams are included, so a missing original cache cannot be overlooked before training. Complete caches can be reused without loading an encoder; missing work may require model downloads and embedding computation.

Extraction records input identity before writing its first shard and checks token counts and finite values before publishing each new shard. Finished shards survive a failure; an unfinished shard may need to be repeated. A printed sequence count is progress, while a published shard is recoverable work. Cache and best-weight files remain local and ignored by Git.

The resulting table distinguishes extraction from reuse. Its GB column reports stored token payload, and the elapsed time describes that particular invocation and its mix of reused and computed work.

In [37]:
written = []
if READ_SAVED_RESULTS:
    print("Extraction skipped. Missing caches or shards are not computed in read-saved mode.")
else:
    started = time.perf_counter()
    for key in BENCHMARKS:
        needed = [e for e in required_encoders(key) if (key, e) in PENDING_CACHE_KEYS]
        if not needed:
            continue
        rows = BENCHMARK_ROWS[key]
        for encoder_key in needed:
            encoder = ENCODERS[encoder_key]
            texts = tokens.inputs_for_rows(encoder, rows["sequence"].tolist())
            cache, computed = tokens.extract(
                encoder, texts, CACHE_ROOT / key / encoder_key, DEVICE,
                chunk_size=CHUNK_SIZE, dtype=DTYPE,
                recovery_evidence=RECOVERY_RECORDS.get(f"{key}/{encoder_key}"),
                verification=CACHE_VERIFICATION,
            )
            try:
                written.append({
                    "benchmark": key, "encoder": encoder.label, "sequences": len(cache),
                    "tokens": cache.meta["total_tokens"], "width": cache.width,
                    "GB": round(cache.meta["bytes"] / 1e9, 2),
                    "this run": "extracted" if computed else "already complete",
                })
            finally:
                cache.close()
    if written:
        display(pd.DataFrame(written).set_index(["benchmark", "encoder"]))
    print(f"Extraction/reuse checks finished in {(time.perf_counter() - started) / 60:.1f} minutes.")


W1102 17:18:46.911000 22092 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: InstaDeepAI/nucleotide-transformer-500m-human-ref
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.decoder.weight    | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


    nt500m: shard 1/94 (1/94 this run)
    nt500m: shard 2/94 (2/94 this run)
    nt500m: shard 3/94 (3/94 this run)
    nt500m: shard 4/94 (4/94 this run)
    nt500m: shard 5/94 (5/94 this run)
    nt500m: shard 6/94 (6/94 this run)
    nt500m: shard 7/94 (7/94 this run)
    nt500m: shard 8/94 (8/94 this run)
    nt500m: shard 9/94 (9/94 this run)
    nt500m: shard 10/94 (10/94 this run)
    nt500m: shard 11/94 (11/94 this run)
    nt500m: shard 12/94 (12/94 this run)
    nt500m: shard 13/94 (13/94 this run)
    nt500m: shard 14/94 (14/94 this run)
    nt500m: shard 15/94 (15/94 this run)
    nt500m: shard 16/94 (16/94 this run)
    nt500m: shard 17/94 (17/94 this run)
    nt500m: shard 18/94 (18/94 this run)
    nt500m: shard 19/94 (19/94 this run)
    nt500m: shard 20/94 (20/94 this run)
    nt500m: shard 21/94 (21/94 this run)
    nt500m: shard 22/94 (22/94 this run)
    nt500m: shard 23/94 (23/94 this run)
    nt500m: shard 24/94 (24/94 this run)
    nt500m: shard 25/94 (25/94 thi

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] HyenaDnaModel LOAD REPORT from: multimolecule/hyenadna-large
Key                 | Status  | 
--------------------+---------+-
pooler.dense.weight | MISSING | 
pooler.dense.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


... (8 outputs left out)

[transformers] BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    protbert: shard 1/94 (1/94 this run)
    protbert: shard 2/94 (2/94 this run)
    protbert: shard 3/94 (3/94 this run)
    protbert: shard 4/94 (4/94 this run)
    protbert: shard 5/94 (5/94 this run)
    protbert: shard 6/94 (6/94 this run)
    protbert: shard 7/94 (7/94 this run)
    protbert: shard 8/94 (8/94 this run)
    protbert: shard 9/94 (9/94 this run)
    protbert: shard 10/94 (10/94 this run)
    protbert: shard 11/94 (11/94 this run)
    protbert: shard 12/94 (12/94 this run)
    protbert: shard 13/94 (13/94 this run)
    protbert: shard 14/94 (14/94 this run)
    protbert: shard 15/94 (15/94 this run)
    protbert: shard 16/94 (16/94 this run)
    protbert: shard 17/94 (17/94 this run)
    protbert: shard 18/94 (18/94 this run)
    protbert: shard 19/94 (19/94 this run)
    protbert: shard 20/94 (20/94 this run)
    protbert: shard 21/94 (21/94 this run)
    protbert: shard 22/94 (22/94 this run)
    protbert: shard 23/94 (23/94 this run)
    protbert: shard 24/94 (24

sequences    tokens  width     GB          this run
benchmark      encoder                                                                      
mrna_stability NT v2 100M multi-species      23929   2642703    512   2.71  already complete
               RNA-FM                        23929  15678663    640  20.07  already complete
               ESM-2 8M                      23929   5226221    320   3.34  already complete
               NT 500M human-ref             23929   2642703   1280   6.77         extracted
               HyenaDNA large                23929  15678663    256   8.03         extracted
               RiNALMo 150M                  23929  15678663    640  20.07         extracted
               ESM-2 35M                     23929   5226221    480   5.02         extracted
               ProtBERT                      23929   5226221   1024  10.70         extracted

Extraction/reuse checks finished in 142.1 minutes.


**Reading the embedding inventory.** The table above describes the cached numerical features available to the prediction models.

| Field | Meaning |
| --- | --- |
| `sequences` | Benchmark rows represented in the cache, including repeated sequences if the benchmark contains them. |
| `tokens` | Total represented positions across those rows. Tokenization differs by encoder, so encoders can represent the same sequences with different token counts. |
| `width` | The number of values in each token's embedding vector. |
| `GB` | Stored token-array payload in decimal gigabytes, calculated from token count, width and dtype size. It excludes archive and metadata overhead and is not a peak RAM or GPU-memory requirement. |
| `this run` | `already complete` means this invocation reused the cache without loading the encoder. `extracted` means it computed missing chunks, which may be only part of the cache. |

Complete embeddings supply the inputs for fitting; they do not show whether prediction models have been trained or how accurately those models predict the benchmark outcome. The next check assembles each encoder triple, followed by the training and interpretation sections.

### Check each triple as training will read it

Open the original streams plus each replacement as a three-stream input. `BenchmarkStreams.open` checks row counts and codon-grid token lengths. The preceding preflight supplies the stronger recorded-input identity checks; token-length checks alone do not prove sequence identity. Rerunning this cell first releases the previous streams' mapped files.

Inspect the encoder names, grid resolutions and widths in the table before fitting. A cache mismatch stops preparation and needs to be resolved before interpreting a fusion comparison.

In [38]:
# Keep no global pool of open token arrays. Each training group owns and closes
# one triple, so completed runs do not require their large caches to be opened.
STREAMS = {}
print("Token streams open only for a pending training group or explicit interval analysis.")


Token streams open only for a pending training group or explicit interval analysis.


---

## Part 2: Fit the encoder comparisons

### Define the training configurations

**Why training follows extraction.** The pretrained encoders have converted sequences into numerical features. Training here keeps those embeddings fixed and learns the prediction head, together with any fusion and grid parameters used by that variant, to predict the benchmark's measured outcome. The pretrained encoders themselves are not fine-tuned.

The comparisons are the scientific purpose: does replacing one encoder improve the overall predictor, and does combining modalities add predictive value beyond the strongest individual encoder in that triple? For example, a replacement RNA encoder could improve both its single-encoder baseline and the fused model. Comparing fusion with the best single tests whether fusion adds predictive value beyond the strongest individual stream; comparing the fused model with the original triple measures the overall effect of the swap.

For each benchmark and replacement, fit the swapped-in encoder alone. When its stream is pooled or upsampled onto the grid, also fit the same encoder on that grid. Then compare concatenation, the gated MIL head and cross-attention using the replacement in its modality's slot.

| Variant in the code | What learns to predict the target | Why include it? |
| --- | --- | --- |
| `unimodal:<slot>` | TextCNN reads one encoder at its native token resolution | Measures what the replacement supports on its own |
| `unimodal-grid:<slot>` | The selected stream is placed on the codon grid before TextCNN | Checks whether changing resolution affects the single-encoder reference |
| `concat` | Features from all three streams are joined at each grid position and projected before TextCNN | Tests combining the representations without a learned modality gate |
| `mil` | A learned gate assigns one weight per modality per sequence, then combines the streams before TextCNN | Tests adaptive weighting across modalities; these weights are not codon-specific |
| `cross` | Each stream attends to the joint three-stream context before the updated streams are merged and read by TextCNN | Tests interactions between positions and modalities |

The gated model's **entropy weight**, `entropy_lambda`, controls a training penalty on how spread out its modality weights are. The implemented loss adds positive entropy, so minimizing that term favors concentrated weights. Phase B reuses the value selected on validation for the original encoders. It does not retune the replacement or establish that a large gate weight means biological importance. See the [fusion implementations](../shared_code/fusion/heads.py) and [training loss](../shared_code/fusion/train.py).

In [ ]:
def open_fit_streams(key, encoder_mapping):
    """Open one triple, releasing earlier caches if a later open/check fails."""
    rows = BENCHMARK_ROWS.get(key)
    if rows is None:
        rows = datasets.blf_benchmark_rows(key)
    spec = datasets.DATASETS[key]
    caches = {}
    try:
        for modality, encoder_key in encoder_mapping.items():
            caches[modality] = tokens.summarize(CACHE_ROOT / key / encoder_key)
        streams = blf.BenchmarkStreams(
            key, rows, caches, spec.task, spec.label_column, spec.split_column,
            encoders=dict(encoder_mapping),
        )
        streams.check()
        return streams
    except BaseException as error:
        for cache in caches.values():
            try:
                cache.close()
            except BaseException as cleanup_error:
                if hasattr(error, "add_note"):
                    error.add_note(f"Cache cleanup also failed: {cleanup_error}")
        raise


def run_pending_group(key, encoder_key, planned_runs):
    """Run only new/resumable fits for one triple; reuse needs no token arrays."""
    pending = [run for run in planned_runs if run.status in ("new", "resume")]
    if not pending:
        return
    streams = open_fit_streams(key, pending[0].encoders)
    primary_error = None
    try:
        for run in pending:
            print(f"{key} / {encoder_key or 'original references'} / {run.variant} / seed {SEED}")
            train.run(
                streams, run.variant, train.TrainConfig(**run.config), DEVICE,
                run.path.parent, tag=run.path.stem,
            )
    except BaseException as error:
        primary_error = error
        raise
    finally:
        try:
            streams.close()
        except BaseException as cleanup_error:
            if primary_error is None:
                raise
            if hasattr(primary_error, "add_note"):
                primary_error.add_note(f"Stream cleanup also failed: {cleanup_error}")


### Run the baselines and fusion heads

**Execution order.** Missing same-seed original references run first: native DNA/RNA/protein predictors, the original RNA grid control and the compared fusion heads. These are listed explicitly in the plan because a replacement must be compared with references fitted at the same seed. Compatible existing references are reused at their actual paths, including historical native-baseline filenames. The replacement sweep then retains its first pass of native/grid singles, concatenation and MIL, followed by cross-attention.

**Reuse and recovery.** The metadata plan checks the full training recipe, fixed selection prerequisites, duplicate identities, destination conflicts and recoverable checkpoints before extraction. It refreshes after extraction. The strict token preflight checks ordered inputs and cache compatibility only for pending fits. A compatible completed result is never fitted again to obtain missing predictions or weights. An incomplete or conflicting publication is preserved and reported instead of overwritten. These checks do not reconstruct unknown historical code or input identity.

**Resources and interruptions.** One triple is opened for a pending training group and closed on completion or failure; the notebook no longer keeps every benchmark/replacement's mappings open. Existing epoch checkpoints retain optimizer, scheduler, random-generator and batch-order state. Compatible checkpoints resume from their saved boundary, so unfinished epochs after that boundary may repeat; completed result records do not. Checkpoints missing required recovery evidence stop in preflight. Numerical trajectories can still vary across hardware.

The elapsed time applies to this invocation. No overnight duration or peak-memory guarantee is implied for the expanded stability workload.


In [40]:
if READ_SAVED_RESULTS:
    print("Training skipped. The execution plan reports reusable, missing and blocked fits.")
else:
    started = time.perf_counter()
    # Recheck destinations after extraction, so a completed canonical or renamed
    # result discovered since preflight is also reused rather than fitted again.
    FIT_PLANS = {key: execution_plan(key) for key in BENCHMARKS}
    for plan in FIT_PLANS.values():
        plan.require_ready()

    # New seeds need same-seed original references as well as replacement fits.
    # Their missing entries are explicit in the plan, never inferred as completed.
    for key, plan in FIT_PLANS.items():
        run_pending_group(key, None, [r for r in plan.runs if r.role == "original"])

    # Preserve the replacement sweep's cheap-first, cross-attention-second order.
    for cross_pass in (False, True):
        for key, plan in FIT_PLANS.items():
            for modality, encoder_key in swaps_for(key):
                group = [r for r in plan.runs
                         if r.role == "replacement" and r.encoder_key == encoder_key
                         and (r.variant == "cross") == cross_pass]
                run_pending_group(key, encoder_key, group)
    print(f"Requested seed finished in {(time.perf_counter() - started) / 3600:.2f} hours.")


mrna_stability / nt500m / unimodal:dna / seed 0
  mrna_stability__unimodal-dna__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 0.9252  val 0.3856  best 0.3856 @ 0
    epoch  10  train loss 0.3137  val 0.5202  best 0.5252 @ 8
    epoch  20  train loss 0.2100  val 0.5253  best 0.5261 @ 17
    epoch  30  train loss 0.2007  val 0.5250  best 0.5261 @ 17
  mrna_stability__unimodal-dna__lambda0.0__ch100__equation__seed0: test 0.5368 (val 0.5261 at epoch 17, 16.9 min)
mrna_stability / nt500m / unimodal-grid:dna / seed 0
  mrna_stability__unimodal-grid-dna__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 0.9209  val 0.3440  best 0.3440 @ 0
    epoch  10  train loss 0.3495  val 0.5323  best 0.5323 @ 10
    epoch  20  train loss 0.2125  val 0.5332  best 0.5332 @ 20
    epoch  30  train loss 0.1954  val 0.5335  best 0.5341 @ 22
    epoch  40  train loss 0.1940  val 0.5336  best 0.5341 @ 22
  mrna_stability__unimodal-grid-dna__lambda0.0__ch100__equation_

---

## Part 3: Interpret the comparisons

For each regression benchmark, training minimizes mean squared error on the stored target scale, while the reported Spearman correlation measures predicted ordering. The encoder comparison therefore asks whether a replacement improves ranking of that benchmark's labels. Higher mRNA-stability labels indicate relatively greater stability; higher CoV-Vac labels indicate greater degradation. Neither score is a measurement of hours of persistence or vaccine performance, and fungal expression concerns RNA abundance rather than directly measured protein production.

For E. coli, accuracy measures how often the predicted low, medium or high expression category agrees with the recorded category. A score gain is an improvement in category prediction, not a proportional increase in protein yield. Compare each replacement with the original encoder on the same benchmark and test rows; differences between benchmarks also reflect their different targets, assays and evaluation tasks.


### Compare each swap with its baselines

Each benchmark table starts with BioLangFusion's original triple, followed by the replacements. The single-encoder scores belong to the three encoders in that row's triple. Seed-0 validation selects the “best single,” including a gridded reading where one is available. Seeds 1 and 2 reuse that selected variant and display its own same-seed test score.

**Two different questions.** Δfusion is a fusion head's test score minus its triple's validation-selected best single. “vs original” compares that fusion head with the same head using the original encoders on the same test rows. A replacement can improve the single encoder without increasing the benefit from fusion; read both differences together.

For example, an illustrative fused score of 0.85 against its own single-encoder reference of 0.84 gives a fusion gain of 0.01. If the original fused model scored 0.79, the same result gives a replacement gain of 0.06. The larger number describes the improvement from changing the encoder; it is not the additional contribution of fusion. These illustrative values are not another experimental result.

The `alone` column is the swapped encoder's native-resolution test score. `best single` is selected using validation, so it can name another stream or a grid baseline, and need not have the highest observed test score. Selecting a new winner after reading the test table would change the evaluation rule. E. coli scores use percent accuracy and differences use percentage points; regression differences use Spearman-correlation units.

The reader selects the configured training seed and full training recipe before constructing each triple. Recorded encoder mappings are checked when present; legacy files without them retain that provenance limit. Duplicate identities are withheld from reading and block computation; unrelated valid scores remain readable. Original native-resolution baselines bypass convolution: a matching convolution label is preferred, with a unique older label accepted when no matching record exists. Grid and fusion fits must match the requested convolution. If required single-encoder references are missing, the best-single score and its fusion gain remain unavailable.

In [41]:
# E. coli accuracy is displayed as a percentage; score differences are percentage points.
SCALE = {"ecoli_proteins": 100.0}
HEADS = ("concat", "mil", "cross")


def swap_runs(key, encoder_key, lam, seed=None):
    """Read only the selected seed and inherited recipe, refusing duplicate identities."""
    modality = next(m for m, e in swaps_for(key) if e == encoder_key)
    report = swap_report(key, modality, encoder_key, SEED if seed is None else seed, lam)
    records = report.records if READ_SAVED_RESULTS else report.require_unique()
    return [train.Result.load(r["path"]) for r in records]


def triple(key, modality=None, encoder_key=None, seed=None):
    """Collect unique variant records for the original or a replacement triple."""
    try:
        lam = original_lambda(key)
    except RuntimeError:
        if not READ_SAVED_RESULTS:
            raise
        # Missing MIL selection does not hide independent single/concat/cross scores.
        lam = None
    runs = {}
    # Keep the validation-selected positive MIL penalty among the original runs.
    for r in original_runs(key, seed):
        name = label(r)
        if name == "mil" and r.config["entropy_lambda"] != lam:
            continue
        runs[name] = r
    if encoder_key is not None:
        # Retain unchanged-slot singles; replace this slot's baselines and compared heads.
        for name in [
            n for n in runs
            if n in HEADS or n.endswith(f":{modality}")
        ]:
            del runs[name]
        for r in swap_runs(key, encoder_key, lam, seed):
            runs[label(r)] = r
    return runs


def best_single(runs, modality=None, encoder_key=None, selection_runs=None):
    """Select on validation only when the required single-encoder references exist."""
    required = {f"unimodal:{m}" for m in ORIGINAL}
    if encoder_key and tokens.grid_resolution(encoder_key) != "codon":
        required.add(f"unimodal-grid:{modality}")
    selection_runs = runs if selection_runs is None else selection_runs
    if not required.issubset(selection_runs):
        return None
    # Available historical grid controls remain eligible; missing native references
    # must not make an incomplete set look like a fully compared "best single."
    singles = {n: r for n, r in selection_runs.items() if n.startswith("unimodal")}
    chosen = max(singles, key=lambda n: singles[n].val_score)
    return chosen if chosen in runs else None

**Build the comparison rows.** `summary` puts the native single-encoder score, the validation-selected reference, each fusion score, and the two kinds of difference in one row per replacement. Missing results remain missing rather than becoming zero scores.

In [42]:
def summary(key):
    """Build one benchmark's score and gain rows from saved results."""
    scale = SCALE.get(key, 1.0)
    original = triple(key)
    rows = []
    # The (None, None) entry supplies the original-triple reference row first.
    for modality, encoder_key in [(None, None), *swaps_for(key)]:
        runs = triple(key, modality, encoder_key)
        selection_runs = (runs if SEED == SELECTION_SEED
                          else triple(key, modality, encoder_key, seed=SELECTION_SEED))
        best = best_single(runs, modality, encoder_key, selection_runs)
        # No replacement-only score exists for the original row; unavailable scores stay NaN.
        row = {
            "slot": modality or "none",
            "encoder": (
                ENCODERS[encoder_key].label if encoder_key else "BioLangFusion's own"
            ),
            "alone": (
                runs.get(f"unimodal:{modality}").test_score * scale
                if modality and f"unimodal:{modality}" in runs else np.nan
            ),
            "best single": best,
            "best single score": (
                runs[best].test_score * scale if best else np.nan
            ),
        }
        for head in HEADS:
            score = runs[head].test_score * scale if head in runs else np.nan
            row[head] = score
            # Distinguish fusion benefit from the overall effect of replacing an encoder.
            row[f"{head} Δfusion"] = score - row["best single score"]
            row[f"{head} vs original"] = (
                score - original[head].test_score * scale
                if head in original and encoder_key else np.nan
            )
        rows.append(row)
    return pd.DataFrame(rows).set_index(["slot", "encoder"])

**Read the benchmark tables.** The following cell reads saved results and displays each benchmark separately. Compare scores within a benchmark, then use the paired intervals below to assess the differences.

In [43]:
for key in BENCHMARKS:
    print(f"\n{key}" + (" (accuracy x 100)" if key in SCALE else " (Spearman)"))
    try:
        display(summary(key).round(3))
    except (OSError, RuntimeError, ValueError, KeyError, TypeError) as missing:
        print("  not ready:", missing)


cov_vac_degradation (Spearman)


alone   best single  best single score  concat  concat Δfusion  concat vs original    mil  mil Δfusion  mil vs original  cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                              
none BioLangFusion's own    NaN  unimodal:rna              0.833   0.801          -0.032                 NaN  0.783       -0.050              NaN  0.832         -0.001                NaN
dna  NT 500M human-ref    0.727  unimodal:rna              0.833   0.753          -0.080              -0.048  0.710       -0.123           -0.073  0.811         -0.022             -0.021
     HyenaDNA large       0.755  unimodal:rna              0.833   0.832          -0.001               0.030  0.797       -0.036            0.014  0.872          0.039              0.040
rna  RiNALMo 150M         0.823  unimodal:rna              0.823   0.806          -0.017               0.004  0.819       -0.004            0.036  0.854          0.031              0.021
     mRNA-FM              0.812  unimodal:rna              0.812   0.759          -0.054              -0.043  0.762       -0.050           -0.021  0.822          0.009             -0.011
     CaLM                 0.844  unimodal:rna              0.844   0.817          -0.027               0.016  0.802       -0.042            0.019  0.828         -0.017             -0.005
prot ESM-2 35M            0.814  unimodal:rna              0.833   0.808          -0.025               0.007  0.793       -0.040            0.010  0.851          0.018              0.018
     ESM-2 150M           0.827  unimodal:rna              0.833   0.802          -0.031               0.001  0.783       -0.050            0.000  0.853          0.020              0.020
     ProtBERT             0.776  unimodal:rna              0.833   0.788          -0.045              -0.013  0.791       -0.043            0.007  0.843          0.010              0.011


fungal_expression (Spearman)


alone        best single  best single score  concat  concat Δfusion  concat vs original    mil  mil Δfusion  mil vs original  cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                                   
none BioLangFusion's own    NaN       unimodal:dna              0.813   0.793          -0.020                 NaN  0.802       -0.011              NaN  0.824          0.011                NaN
dna  NT 500M human-ref    0.740  unimodal-grid:rna              0.794   0.794           0.000               0.002  0.791       -0.003           -0.011  0.810          0.016             -0.014
     HyenaDNA large       0.737  unimodal-grid:rna              0.794   0.786          -0.007              -0.006  0.780       -0.014           -0.022  0.804          0.010             -0.020
rna  RiNALMo 150M         0.844       unimodal:rna              0.844   0.846           0.002               0.054  0.831       -0.013            0.029  0.847          0.003              0.023
     mRNA-FM              0.852       unimodal:rna              0.852   0.853           0.001               0.061  0.839       -0.013            0.037  0.840         -0.012              0.017
     CaLM                 0.851       unimodal:rna              0.851   0.848          -0.003               0.056  0.848       -0.003            0.046  0.857          0.006              0.034
prot ESM-2 35M            0.696       unimodal:dna              0.813   0.795          -0.018               0.002  0.780       -0.033           -0.022  0.819          0.006             -0.005
     ESM-2 150M           0.729       unimodal:dna              0.813   0.784          -0.028              -0.008  0.784       -0.028           -0.018  0.811         -0.002             -0.013
     ProtBERT             0.761      unimodal:prot              0.761   0.805           0.044               0.012  0.800        0.039           -0.002  0.816          0.054             -0.008


ecoli_proteins (accuracy x 100)


alone        best single  best single score  concat  concat Δfusion  concat vs original     mil  mil Δfusion  mil vs original   cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                                      
none BioLangFusion's own     NaN      unimodal:prot             50.140  60.674          10.534                 NaN  54.354        4.213              NaN  53.792          3.652                NaN
dna  NT 500M human-ref    47.472      unimodal:prot             50.140  57.865           7.725              -2.809  54.213        4.073           -0.140  55.337          5.197              1.545
     HyenaDNA large       50.421      unimodal:prot             50.140  58.006           7.865              -2.669  53.090        2.949           -1.264  51.966          1.826             -1.826
rna  RiNALMo 150M         57.303  unimodal-grid:rna             56.039  59.972           3.933              -0.702  62.500        6.461            8.146  54.354         -1.685              0.562
     mRNA-FM              58.708       unimodal:rna             58.708  58.989           0.281              -1.685  53.090       -5.618           -1.264  54.775         -3.933              0.983
     CaLM                 57.584       unimodal:rna             57.584  57.725           0.140              -2.949  56.742       -0.843            2.388  53.933         -3.652              0.140
prot ESM-2 35M            52.107      unimodal:prot             52.107  57.725           5.618              -2.949  53.230        1.124           -1.124  53.230          1.124             -0.562
     ESM-2 150M           53.933      unimodal:prot             53.933  57.725           3.792              -2.949  55.478        1.545            1.124  54.635          0.702              0.843
     ProtBERT             52.528      unimodal:prot             52.528  54.213           1.685              -6.461  56.461        3.933            2.107  54.635          2.107              0.843


mrna_stability (Spearman)


alone        best single  best single score  concat  concat Δfusion  concat vs original    mil  mil Δfusion  mil vs original  cross  cross Δfusion  cross vs original
slot encoder                                                                                                                                                                                   
none BioLangFusion's own    NaN       unimodal:rna              0.552   0.558           0.007                 NaN  0.551       -0.001              NaN  0.553          0.001                NaN
dna  NT 500M human-ref    0.537       unimodal:rna              0.552   0.552          -0.000              -0.007  0.553        0.002            0.003  0.545         -0.006             -0.007
     HyenaDNA large       0.523       unimodal:rna              0.552   0.557           0.005              -0.001  0.539       -0.012           -0.011  0.548         -0.004             -0.005
rna  RiNALMo 150M         0.560  unimodal-grid:rna              0.556   0.556           0.000              -0.002  0.564        0.008            0.014  0.558          0.003              0.006
     mRNA-FM              0.565       unimodal:rna              0.565   0.568           0.003               0.010  0.565       -0.000            0.014  0.562         -0.003              0.009
     CaLM                 0.559       unimodal:rna              0.559   0.561           0.002               0.003  0.561        0.002            0.010  0.548         -0.010             -0.004
prot ESM-2 35M            0.535       unimodal:rna              0.552   0.554           0.003              -0.004  0.551       -0.000            0.001  0.541         -0.011             -0.012
     ESM-2 150M           0.536       unimodal:rna              0.552   0.559           0.008               0.001  0.549       -0.003           -0.002  0.559          0.007              0.006
     ProtBERT             0.543      unimodal:prot              0.543   0.558           0.015              -0.001  0.550        0.008           -0.000  0.551          0.009             -0.001

**What the saved score tables suggest.** The fungal RNA replacements have strong single-encoder scores, while their extra gains from fusion are small or negative in several rows. CoV-Vac and E. coli also contain positive fusion differences for particular replacement/head combinations. These patterns answer different questions from a universal encoder ranking: both the biological target and the downstream model matter.

Point differences alone do not tell us which directions the evaluated test rows support. The next section compares each fitted model with its declared reference using those same rows, before returning to the worked RNA-FM/mRNA-FM example.

### Assess what the paired differences resolve

The paired bootstrap compares predictions on the same test rows. For each head and replacement, it estimates the difference from the original-encoder head and the difference from the replacement triple's best single. Exact duplicate sequences are grouped during resampling.

**Interpretation.** Read each interval against zero. An interval spanning zero leaves the direction unresolved under this procedure; it does not establish equivalence. Test-set sizes differ across the four benchmarks, so neither a universal row-count range nor a fixed 0.02 score threshold is justified. These intervals describe uncertainty from the evaluated rows, not variation across training seeds or protection against train/test sequence overlap.

Here, `resolve` requests 2,000 resamples with bootstrap seed 0. This seed controls resampling; it is separate in purpose from the training seed, even though both are numerically zero. Despite the helper name `seed_averaged_bootstrap`, each model is supplied as a list containing one saved fit in this notebook.

| Output | How to read it |
| --- | --- |
| `difference` | First named model's score minus its reference on the evaluated test rows |
| `ci_low`, `ci_high` | 2.5th and 97.5th percentiles of the paired resampled differences |
| `first_wins` | Fraction of resamples with a positive difference; not a p-value or the probability that a biological hypothesis is true |
| `resolved` | Whether the interval excludes zero in either direction; `yes` can therefore mean a loss as well as a gain |

These are individual comparison intervals without a correction for examining many swaps and heads. They describe uncertainty conditional on the saved fitted models and evaluation procedure.

Before resampling, the reader requires saved predictions and checks their row counts and saved target order against the current test targets at the loader's label precision (float32 for regression, int64 for classification). The bootstrap itself keeps its existing target array. This detects missing or mismatched arrays; equal target values alone do not prove sequence identity where row-level provenance was never recorded. Score-only historical results remain usable in the preceding score table.

In [44]:
def resolve(key, draws=2000):
    """Compare saved predictions on paired, sequence-grouped test resamples."""
    # Prediction pairing needs the benchmark's labels/order, not any token arrays.
    rows = BENCHMARK_ROWS.get(key)
    if rows is None:
        rows = datasets.blf_benchmark_rows(key)
    spec = datasets.DATASETS[key]
    test = np.flatnonzero(rows[spec.split_column].to_numpy() == "test")
    truth = rows[spec.label_column].to_numpy()[test]
    task = spec.task
    target_dtype = np.int64 if task == "classification" else np.float32
    saved_target_order = np.asarray(truth, dtype=target_dtype).tolist()
    groups = rows["sequence"].to_numpy()[test]
    n_outputs = int(rows[spec.label_column].max()) + 1 if task == "classification" else None
    original = triple(key)
    predictions, comparisons, records = {}, [], {}

    for name, r in original.items():
        records[f"original {name}"] = r
    for modality, encoder_key in swaps_for(key):
        runs = triple(key, modality, encoder_key)
        selection_runs = (runs if SEED == SELECTION_SEED
                          else triple(key, modality, encoder_key, seed=SELECTION_SEED))
        best = best_single(runs, modality, encoder_key, selection_runs)
        for name, r in runs.items():
            records[f"{encoder_key} {name}"] = r
        for head in HEADS:
            if head not in runs or head not in original:
                continue
            # Each tuple names the displayed comparison, first predictor, and reference.
            comparisons.append((
                f"{encoder_key} {head} vs original",
                f"{encoder_key} {head}",
                f"original {head}",
            ))
            if best is not None:
                comparisons.append((
                    f"{encoder_key} {head} vs best single ({best})",
                    f"{encoder_key} {head}",
                    f"{encoder_key} {best}",
                ))
    # Validate only records used by a comparison: score-only historical baselines
    # remain readable, but missing/misordered predictions cannot support a paired interval.
    needed = {name for _, first, second in comparisons for name in (first, second)}
    for name in sorted(needed):
        r = records[name]
        issue = swap_results.prediction_issue(
            vars(r), saved_target_order, task=task,
            n_classes=n_outputs,
        )
        if issue:
            raise ValueError(f"{key}, {name}: {issue}; paired comparison not calculated")
        predictions[name] = [np.asarray(r.test_prediction)]
    if not comparisons:
        raise ValueError(f"{key}: no compatible paired comparisons; inspect result coverage")

    # Each prediction list contains one fitted run; this is not a multi-seed average here.
    out = pd.DataFrame(
        train.seed_averaged_bootstrap(
            task,
            truth,
            predictions,
            comparisons,
            groups=groups,
            draws=draws,
            seed=0,
        )
    )

    # Convert score differences to percentage points for classification, not first_wins.
    scale = SCALE.get(key, 1.0)
    for column in ("difference", "ci_low", "ci_high"):
        out[column] = out[column] * scale
    # Excluding zero resolves either a gain or a loss; retain the difference's sign.
    out["resolved"] = np.where(
        (out["ci_low"] > 0) | (out["ci_high"] < 0),
        "yes",
        "no",
    )
    return out.set_index("comparison")[[
        "difference", "ci_low", "ci_high", "first_wins", "resolved",
    ]]

**Read or calculate paired intervals.** The fit-completion path leaves `RUN_BOOTSTRAP = False`, including when `READ_SAVED_RESULTS` is false. No standalone full-sweep interval artifact is configured here. Retained outputs and the linked run record document earlier intervals; they are not reconstructed from rounded display values. Explicitly enabling bootstrap resamples saved predictions on the same test rows and uses the fixed seed-0 baseline choice, without opening token caches or refitting prediction models. It calculates an interval for the selected training seed, not an average across three seeds.


In [45]:
RESOLVED = {}
if READ_SAVED_RESULTS or not RUN_BOOTSTRAP:
    print("Paired intervals not recalculated. RUN_BOOTSTRAP is separate from completing fits; "
          "retained interval outputs remain historical.")
else:
    for key in BENCHMARKS:
        try:
            RESOLVED[key] = resolve(key)
        except (RuntimeError, ValueError, KeyError) as missing:
            print(f"{key}: not ready ({missing})")
            continue
        print(f"\n{key}")
        display(RESOLVED[key].round(3))


Paired intervals not recalculated. RUN_BOOTSTRAP is separate from completing fits; retained interval outputs remain historical.


### What the saved comparisons establish

The following examples summarize the retained score and interval tables and the [October 27 readout](../../docs/run-records.md#2026-10-27-phaseb-complete-readout-all-eight-swaps-written-up-from-the-saved-notebook-outputs-no-new-fits). They are readings of existing seed-0 results, not new fits or recalculated intervals.

| Comparison | Saved observation | What it supports and what it leaves open |
| --- | --- | --- |
| CoV-Vac, HyenaDNA with cross-attention | Spearman 0.872; gain over the selected RNA-FM single of +0.039 [0.012, 0.066] | A positive fusion difference for this replacement/head on chemical-degradation labels; it does not establish a translation mechanism for these RNA fragments |
| Fungal expression, ProtBERT with cross-attention | Spearman 0.816; gain over the validation-selected ProtBERT single of +0.054 [0.033, 0.076] | A positive difference against the declared 0.761 reference; the retained NT v2 test score is 0.813, so this is not evidence of a large margin over every single encoder |
| E. coli, RiNALMo with gated MIL | Accuracy 62.500%; gain over the selected gridded RNA single of +6.461 [3.371, 9.551] percentage points | A positive fusion difference for expression-category prediction; the selected reference is the grid baseline, not RiNALMo's native-resolution score |
| mRNA stability, the configured three replacements | Every displayed fusion-versus-best-single interval includes zero | No resolved extra fusion benefit in this saved subset; it neither establishes equivalence nor covers the other five replacements |

The fungal ProtBERT example also distinguishes a larger fusion gain from a higher absolute score: the mRNA-FM concatenated model scores 0.853, above ProtBERT cross-attention's 0.816, but adds little over mRNA-FM alone. Keep the declared validation selection when interpreting these differences rather than replacing it with a test-selected reference.

These cases qualify a simple story that fusion never helps or that codon pretraining always wins. The evidence is specific to an encoder, head, target, reference, and seed. The worked RNA-FM/mRNA-FM comparison below examines those distinctions in more detail.

### 3. What the saved RNA-FM and mRNA-FM results show

The following reading uses the retained seed-0 outputs above and the [October 27 saved-result readout](../../docs/run-records.md#2026-10-27-phaseb-complete-readout-all-eight-swaps-written-up-from-the-saved-notebook-outputs-no-new-fits). It is not a new fit or recomputation. “Native” means the single-encoder predictor reads that encoder's token sequence before grid pooling; the RNA-FM grid control reads states averaged in threes.

| Benchmark and metric | RNA-FM native | RNA-FM grid control | mRNA-FM native | Concatenation gain over mRNA-FM alone [saved paired interval] |
| --- | ---: | ---: | ---: | --- |
| CoV-Vac degradation, Spearman | 0.833 | 0.823 | 0.812 | −0.054 [−0.088, −0.019] |
| Fungal expression, Spearman | 0.722 | 0.794 | 0.852 | +0.001 [−0.014, +0.016] |
| E. coli protein-expression class, accuracy | 50.70% | 49.30% | 58.71% | +0.28 percentage points [−2.95, +3.51] |
| mRNA stability, Spearman | 0.552 | 0.547 | 0.565 | +0.003 [−0.003, +0.010] |

These are within-benchmark comparisons, not one common scale across assays. The fungal native RNA-FM value comes from the historical Stage 3 record labeled `appendix`; the current reader accepts native unimodal records under either convolution label because they do not use the grid's transposed convolution. That selection rule does not establish complete historical producer identity. The grid control also shows why attributing the entire native-score difference to pretraining would be too strong.

**A stronger replacement is not the same finding as an added fusion benefit.** Fungal concatenation with mRNA-FM scores 0.853, compared with 0.793 for the original triple: the saved difference is +0.061 [0.034, 0.087]. But mRNA-FM alone scores 0.852, and the same fused model's gain over it is +0.001 [−0.014, 0.016]. The replacement improves this fitted pipeline without resolving an additional benefit from the other two streams.

mRNA-FM has higher native single-encoder scores on fungal expression, E. coli and stability, but lower scores on CoV-Vac. CoV-Vac's inputs are designed RNA fragments, not verified CDS; grouping them into triplets does not establish a biological reading frame. Its chemical-degradation target also differs from cellular expression and standardized stability. See the [target definitions](../../docs/methods/data-and-targets.md#cov-vac-chemical-degradation-of-designed-rna).

Across these four mRNA-FM replacement triples, none of the saved concatenation, gated or cross-attention intervals resolves a positive gain over the validation-selected single-encoder reference, which is mRNA-FM in each case. Some intervals include zero and some show a loss; an unresolved interval does not establish equivalence. These intervals resample test-sequence groups for one fitted seed. They do not measure training-seed variability or remove the known stability train/test overlap.

### 4. What remains unexplained, and how it informs fusion

The results are consistent with mRNA-FM making useful coding-sequence features accessible to the predictor. They do not identify whether the advantage comes from its corpus, codon grouping, capacity, wider representations, predictor dimensions or their combination. Likewise, a small fusion gain can reflect overlapping useful representations, an ineffective fusion head, the inherited tuning recipe or limited evidence. It does not show that the additional encoder contains no useful information.

This gives the next interpretation a concrete starting point: what target-relevant information could the other streams provide beyond a strong RNA representation? For example, a protein representation could expose amino-acid properties associated with the expression assay, while a nucleotide representation preserves synonymous variation. Both are views derived from the same supplied sequence here. Their biological relevance and incremental predictive value require evidence; simply naming them different modalities does not establish complementarity.

Comparable training-seed repeats would test whether the observed pattern persists. The existing RNA-FM grid control helps examine token resolution, and the paired comparisons distinguish improvement over the original pipeline from improvement over the selected single encoder. A subsequent combination should be motivated by a specific remaining information gap and a falsifiable comparison, including the possibility of no gain. This discussion proposes explanations to investigate; it does not select a new dataset or change the current experiment.

The following alignment section adds another view of representation relationships. Its measurements come from the Stage 1 stability sample, so agreement with this interpretation would remain sample-specific rather than explain every benchmark's outcome.

### Relate the comparisons to Stage 1 alignment

The Stage 2 decision is to report alignment beside fusion results. A replacement changes two of the triple's three encoder pairs. The table below gives Stage 1's raw and composition-controlled alignment for those pairs, using its 981-sequence mRNA stability sample, where all twelve encoders were measured together on coding sequences.

**Scope of the evidence.** These are measurements on that Stage 1 sample. They supply the frozen-encoder prediction being examined under H2; they are not newly measured alignment on each fusion benchmark.

**Open discrepancy.** The table reads the [Stage 1 capture](../../docs/figures/stability/results.json). Its raw CKA agrees with the saved Stage 1 notebook, but some composition-controlled values differ: RNA-FM with RiNALMo is 0.486 in the capture and 0.378 in the notebook. As recorded in the [run records](../../docs/run-records.md), that discrepancy remains unresolved. Treat the controlled values as provisional.

In [46]:
alignment = []
try:
    # Reuse the Stage 1 capture; no alignment is recomputed for these benchmarks.
    stage1 = json.loads(STAGE1_ALIGNMENT.read_text())


    def matrix(name):
        """Restore encoder-labeled axes from the saved JSON matrix."""
        m = stage1["similarity"][name]
        return pd.DataFrame(m["data"], index=m["index"], columns=m["columns"])


    cka, cka_controlled = matrix("Linear CKA"), matrix("Linear CKA after control")
    alignment = []
    # A swap may appear on several benchmarks, but this shared Stage 1 lookup is needed once.
    for modality, encoder_key in sorted({s for b in BENCHMARKS for s in swaps_for(b)}):
        triple_keys = {**ORIGINAL, modality: encoder_key}
        # Read similarity to each of the two encoders retained in the triple.
        for other in (m for m in ORIGINAL if m != modality):
            a, b = encoder_key, triple_keys[other]
            alignment.append({
                "swap": ENCODERS[encoder_key].label,
                "pair": f"{ENCODERS[a].label} + {ENCODERS[b].label}",
                "CKA": cka.loc[a, b],
                "CKA after composition control": cka_controlled.loc[a, b],
            })
        # Also compare the replacement with the original encoder it displaced.
        alignment.append({
            "swap": ENCODERS[encoder_key].label,
            "pair": (
                f"{ENCODERS[encoder_key].label} vs the "
                f"{ENCODERS[ORIGINAL[modality]].label} it replaces"
            ),
            "CKA": cka.loc[encoder_key, ORIGINAL[modality]],
            "CKA after composition control": cka_controlled.loc[encoder_key, ORIGINAL[modality]],
        })
    display(pd.DataFrame(alignment).set_index(["swap", "pair"]).round(3))
except (OSError, ValueError, KeyError, TypeError) as exc:
    if not READ_SAVED_RESULTS:
        raise
    print(f"Stage 1 alignment unavailable: {exc}. No alignment was recalculated.")


CKA  CKA after composition control
swap              pair                                                                                    
HyenaDNA large    HyenaDNA large + RNA-FM                             0.398                          0.228
                  HyenaDNA large + ESM-2 8M                           0.161                          0.109
                  HyenaDNA large vs the NT v2 100M multi-species ...  0.736                          0.117
NT 500M human-ref NT 500M human-ref + RNA-FM                          0.530                          0.196
                  NT 500M human-ref + ESM-2 8M                        0.163                          0.126
                  NT 500M human-ref vs the NT v2 100M multi-speci...  0.398                          0.064
ESM-2 150M        ESM-2 150M + NT v2 100M multi-species               0.067                          0.030
                  ESM-2 150M + RNA-FM                                 0.200                          0.104
                  ESM-2 150M vs the ESM-2 8M it replaces              0.553                          0.391
ESM-2 35M         ESM-2 35M + NT v2 100M multi-species                0.083                          0.045
                  ESM-2 35M + RNA-FM                                  0.199                          0.175
                  ESM-2 35M vs the ESM-2 8M it replaces               0.744                          0.729
ProtBERT          ProtBERT + NT v2 100M multi-species                 0.058                          0.027
                  ProtBERT + RNA-FM                                   0.116                          0.083
                  ProtBERT vs the ESM-2 8M it replaces                0.199                          0.085
CaLM              CaLM + NT v2 100M multi-species                     0.183                          0.058
                  CaLM + ESM-2 8M                                     0.433                          0.421
                  CaLM vs the RNA-FM it replaces                      0.632                          0.194
mRNA-FM           mRNA-FM + NT v2 100M multi-species                  0.030                          0.044
                  mRNA-FM + ESM-2 8M                                  0.046                          0.106
                  mRNA-FM vs the RNA-FM it replaces                   0.126                          0.206
RiNALMo 150M      RiNALMo 150M + NT v2 100M multi-species             0.209                          0.089
                  RiNALMo 150M + ESM-2 8M                             0.339                          0.227
                  RiNALMo 150M vs the RNA-FM it replaces              0.809                          0.486

### What this investigation supports

Replacing an encoder can improve prediction under the retained BioLangFusion recipe, but improving the fused model and demonstrating an extra benefit from fusion are different outcomes. The fungal mRNA-FM comparison illustrates the first: much of the improvement is already available from the replacement alone. CoV-Vac with HyenaDNA cross-attention and E. coli with RiNALMo gated fusion supply positive paired differences against their declared single-encoder references, so the saved evidence does not support dismissing fusion across the board.

The differences do not isolate why a representation helps. Pretraining populations, token resolution, encoder capacity, the predictor's input dimensions, baseline selection, and inherited tuning remain possible contributors. Stage 1 similarity supplies a reason to ask about complementarity, but these tables do not establish a general relationship between alignment and fusion gain, particularly while the controlled-CKA discrepancy remains unresolved.

The plan's reproducibility criterion remains open: this notebook's replacement comparisons use one training seed. Paired test-row intervals do not supply missing training repeats or remove overlap in the inherited stability split. The evidence therefore identifies specific comparisons worth repeating and interpreting, rather than a universal winning encoder or fusion method.

A later fusion design should follow a concrete question about what information another stream adds for the target. Training-seed repeats, the remaining stability replacements, and the plan's later two-encoder replacements are separate extensions beyond the current sweep.

## Choices and limitations

These choices define the present comparison in addition to Phase A's settings.

| Choice | Reason | Interpretation limit |
| --- | --- | --- |
| One seed per replacement | Keeps the initial sweep bounded | Bootstrap intervals do not measure training-seed variation; reproducibility needs further evidence |
| Reuse the original triple's gated-head entropy weight | Holds this tuning choice fixed across replacements | A replacement may behave differently at its own validation-selected weight |
| Leave DNABERT-2 outside the sweep | Variable-length byte-pair tokens need a fixed grid adapter | This DNA candidate remains untested here |
| Add RiNALMo and ESM-2 35M | Test replacements that Stage 1 found similar to the originals | Similarity motivates an H2 comparison; it does not guarantee little predictive change |
| Use alignment from the Stage 1 stability sample | Provides the existing twelve-encoder comparison | Values are sample-specific, and the controlled-CKA discrepancy remains open |
| Select three replacements for stability | Prioritize the mRFP-motivated RNA comparison, another codon encoder and a larger protein encoder | Five replacements remain untested there; older storage estimates are not verified requirements |
| Defer two-encoder replacements | Candidate triples depend on the single-swap evidence | This sweep does not complete the plan's later triple comparisons |